In [13]:
# Import libraries
import pandas as pd
import numpy as np

In [14]:
# Load data
path = "D:/D/Projects/Healthcare Analytics/data/raw"

admissions = pd.read_csv(path+'/admissions.csv')
claims = pd.read_csv(path+'/claims.csv')
patients = pd.read_csv(path+'/patients.csv')
readmissions = pd.read_csv(path+'/readmissions.csv')

CREATE DATA QUALITY FLAGS

In [15]:
# Convert data column firt, try to do it first
admissions['admit_date'] = pd.to_datetime(admissions['admit_date'])
admissions['discharge_date'] = pd.to_datetime(admissions['discharge_date'])

In [16]:
# Start generate flag for los, dignosis, billing, claims
# Flag LOS data, and count negative day stay because it is impossible dtays
admissions['negative_los_flag'] = admissions['los_days']<0
print('Negative LOS count: ', (admissions['los_days']<0).sum())

# Date Logic error (litle different than negative los)
admissions['date_logic_error_flag'] = (admissions['discharge_date'] < admissions['admit_date'])
print(admissions['date_logic_error_flag'].sum())

# Try to check LOS mismatch
computed_los = (
    admissions['discharge_date'] - admissions['admit_date']
).dt.days

admissions['los_mismatch_flag'] = (
    admissions['los_days'] != computed_los
)
print("LOS mismatch count: ", admissions['los_mismatch_flag'].sum())

Negative LOS count:  2000
2000
LOS mismatch count:  3974


In [17]:
# Orphan patient records
admissions['orphan_patient_flag'] = (
    ~admissions['patient_id'].isin(patients['patient_id'])
)

print("Orphan admissions count: ", admissions['orphan_patient_flag'].sum())

Orphan admissions count:  1000


In [18]:
# This enables:

# filtering safe datasets
# ML-ready datasets
# dashboard-safe subsets
# audit transparency
# reproducibility

# This is enterprise analytics thinking

FINANCIAL INTEGRITY VIOLATION

In [19]:
# Flag for billed amount and approve amount
claims['approved_exceeds_billed_flag'] = (
    claims['approved_amount'] > claims['billed_amount']
)
print("Approved Amount Flag Count: ", claims['approved_exceeds_billed_flag'].sum())

# Payment timeline violation
claims['submission_date'] = pd.to_datetime(claims['submission_date'])
claims['payment_date'] = pd.to_datetime(claims['payment_date'])

claims['payment_before_submission_flag'] = (
    claims['submission_date'] > claims['payment_date']
)
print("Paayment before claim submission count: ", claims['payment_before_submission_flag'].sum())


Approved Amount Flag Count:  4967
Paayment before claim submission count:  501


In [20]:
# Missing payment date when status is approved
claims['missing_payment_date_flag'] = (
    (claims['claim_status'] == "Approved") &
    (claims['payment_date'].isna())
)
print("Missing payment date for approved claims count: ", claims['missing_payment_date_flag'].sum())

# Payment exist but claim not approved
claims['payment_without_approval_flag'] = (
    claims["payment_date"].notna() &
    ~claims["claim_status"].isin(["Approved", "Partial"])
)
print("Payment Exist even claims pending count: ", claims['payment_without_approval_flag'].sum())

# Negative amount in billed
claims['negative_amount_flag'] = (
    (claims['billed_amount'] < 0) |
    (claims['approved_amount'] < 0)
)
print("Negative finacial count: ", claims['negative_amount_flag'].sum())

Missing payment date for approved claims count:  0
Payment Exist even claims pending count:  1966
Negative finacial count:  2000


In [21]:
# Claims flaged columns
claims[[
    "approved_exceeds_billed_flag",
    "payment_before_submission_flag",
    "negative_amount_flag",
    "missing_payment_date_flag",
    "payment_without_approval_flag"
]].sum()

approved_exceeds_billed_flag      4967
payment_before_submission_flag     501
negative_amount_flag              2000
missing_payment_date_flag            0
payment_without_approval_flag     1966
dtype: int64

DETERMINISTIC REPAIR (FIRST REAL CLEANING STEP)

In [22]:
# RULE
# Repair what can be repaired safely, flag what cannot be repaired.

In [23]:
# Repair LOS - never drop this step.
admissions['computed_los'] = (
    admissions['discharge_date'] - admissions['admit_date']
).dt.days

# repair los data
admissions['los_days_corrected'] = admissions['los_days']
admissions.loc[
    admissions['los_mismatch_flag'],
    'los_days_corrected'
] = admissions['computed_los']

# Analytics-safe LOS field
admissions["los_days_analytics"] = admissions["computed_los"]

admissions.loc[
    admissions["date_logic_error_flag"],
    "los_days_analytics"
] = np.nan

print("Negative analytics LOS:", (admissions["los_days_analytics"] < 0).sum())
print("Missing analytics LOS:", admissions["los_days_analytics"].isna().sum())

Negative analytics LOS: 0
Missing analytics LOS: 2000


In [24]:
# Repairing Diagnosis code
# Preserve original diagnosis code
admissions['diagnosis_code_original'] = admissions['diagnosis_code']
admissions['diagnosis_desc_original'] = admissions['diagnosis_desc']

# Perform standardization on diagnosis data to clea it. 
icd_pattern = r"^[A-Z][0-9]{2}(\.[A-Z0-9]{1,3})?$"

# Clean diagnosis code formating only
admissions['diagnosis_code_cleaned'] = (
    admissions['diagnosis_code']
    .astype("string")
    .str.strip()
    .str.upper()
)

# Clean diagnosis description formating
admissions['diagnosis_desc_cleaned'] = (
    admissions['diagnosis_desc']
    .astype("string")
    .str.strip()
)

# missing diagnosis codes with ICD format check
admissions['diagnosis_missing_flag'] = admissions['diagnosis_code'].isna()


admissions['diagnosis_invalid_format_flag'] = (
    admissions['diagnosis_code'].notna() &
    ~admissions['diagnosis_code_cleaned'].str.match(icd_pattern, na=False)
)

# Suspicious diagnosis description
admissions['diagnosis_desc_invalid_flag'] = (
    admissions['diagnosis_desc']
    .astype("string")
    .str.strip()
    .str.upper()
    .isin(["INVALID", "N/A", "MISSING", ""])
)

# create diagnosis usable flag for analytics
admissions["diagnosis_usable_flag"] = ~(
    admissions["diagnosis_missing_flag"] |
    admissions["diagnosis_invalid_format_flag"] |
    admissions["diagnosis_desc_invalid_flag"]
)
print("Diagnosis usable row: ", admissions['diagnosis_usable_flag'].sum())
print("Diagnosis unusable row: ", (~admissions['diagnosis_usable_flag']).sum())


Diagnosis usable row:  196000
Diagnosis unusable row:  4000


In [25]:
# Final flagged data
admissions[
[
"negative_los_flag",
"date_logic_error_flag",
"los_mismatch_flag",
"diagnosis_missing_flag",
"diagnosis_invalid_format_flag",
"diagnosis_desc_invalid_flag",
"orphan_patient_flag"
]
].sum()

negative_los_flag                2000
date_logic_error_flag            2000
los_mismatch_flag                3974
diagnosis_missing_flag            512
diagnosis_invalid_format_flag    3000
diagnosis_desc_invalid_flag      4000
orphan_patient_flag              1000
dtype: Int64

In [26]:
# Orphan admission handling
admissions['patient_link_usable_flag'] = ~admissions['orphan_patient_flag']

print("Linkable admissions:", admissions["patient_link_usable_flag"].sum())
print("Non-linkable admissions:", (~admissions["patient_link_usable_flag"]).sum())

Linkable admissions: 199000
Non-linkable admissions: 1000


In [27]:
# WHO to update           WHAT column        WHAT value to put in
# ─────────────────────   ──────────────     ───────────────────
# claims.loc[flag==True,  "approved_          = claims["billed_
#                          amount_corrected"]    amount"]

In [28]:
# Claim file rearing beging
claims['billed_amount_original'] = claims['billed_amount']
claims['aprroved_amount_original'] = claims['approved_amount']

# create corrected aprroved amount
claims['approved_amount_corrected'] = claims['approved_amount']
claims.loc[
    claims['approved_exceeds_billed_flag'],
    'approved_amount_corrected'
] = claims['billed_amount']

# Count mismatch rows afer cleannign
print((
    claims['approved_amount_corrected'] > claims['billed_amount']
).sum())

# Hnadle negative finacial values safely
claims['finacial_amount_usable_flag'] = ~claims['negative_amount_flag']
print("Usable finacial rows: ", claims['finacial_amount_usable_flag'].sum())

0
Usable finacial rows:  198000


In [29]:
# Normalization of the claim status
claims['claim_status_original'] = claims['claim_status']

claims['claim_status_cleaned'] = (
    claims['claim_status']
    .astype("string")
    .str.strip()
    .str.upper()
    .str.title()
)

claims['claim_status_cleaned'].value_counts()

claim_status_cleaned
Approved    98245
Denied      39235
Pending     38966
Partial     19554
Name: count, dtype: Int64

In [30]:
# Normalization of payer
claims['payer_original'] = claims['payer']

claims['payer_cleaned'] = (
    claims['payer']
    .astype("string")
    .str.strip()
    .str.upper()
    .str.title()
)

claims['payer_cleaned'].value_counts()

payer_cleaned
Medicaid        28692
Cigna           28630
Aetna           28608
Medicare        28591
Self-Pay        28582
Unitedhealth    28532
Bluecross       28365
Name: count, dtype: Int64

In [31]:
# Create claims financial usability flag
claims['financial_record_usable_flag'] = ~(
    claims['negative_amount_flag'] | claims['approved_exceeds_billed_flag']
)
claims['financial_record_usable_flag'].value_counts()

financial_record_usable_flag
True     195033
False      4967
Name: count, dtype: int64

In [32]:
# Create claim workflow usable flag
claims['workflow_record_usable_flag'] = ~(
    claims['payment_before_submission_flag'] |
    claims['payment_without_approval_flag'] |
    claims['missing_payment_date_flag']
)
claims['workflow_record_usable_flag'].value_counts()

workflow_record_usable_flag
True     197542
False      2458
Name: count, dtype: int64

In [33]:
admissions["admission_record_analytics_ready_flag"] = (
    admissions["diagnosis_usable_flag"] &
    admissions["patient_link_usable_flag"] &
    admissions["los_days_analytics"].notna()
)
admissions["admission_record_analytics_ready_flag"].value_counts()

admission_record_analytics_ready_flag
True     193072
False      6928
Name: count, dtype: Int64

In [34]:
claims["claim_record_analytics_ready_flag"] = (
    claims["financial_record_usable_flag"] &
    claims["workflow_record_usable_flag"]
)

print("Analytics-ready claims:",
      claims["claim_record_analytics_ready_flag"].sum())

print(claims["claim_record_analytics_ready_flag"].value_counts())

Analytics-ready claims: 192638
claim_record_analytics_ready_flag
True     192638
False      7362
Name: count, dtype: int64


EXPORT CLEAN ANALYTICS DATASETS WITH ORIGINAL DATA

In [35]:
# Export dataset
cleaned_dataset_path = "D:/D/Projects/Healthcare Analytics/data/clean"
admissions.to_csv(cleaned_dataset_path+'/admissions_clean.csv', index=False)
claims.to_csv(cleaned_dataset_path+'/claims_clean.csv', index=False)
patients.to_csv(cleaned_dataset_path+'/patients_clean.csv', index=False)
readmissions.to_csv(cleaned_dataset_path+'/readmissions_clean.csv', index=False)


PHASE 4: CLAIM STATUS DISTRIBUTION

In [36]:
# Let's build KPI for every claims
claims['claim_status_cleaned'].value_counts()

claim_status_cleaned
Approved    98245
Denied      39235
Pending     38966
Partial     19554
Name: count, dtype: Int64

In [38]:
#  Clculating Approval rate, denial rate, pending rate
approval_rate = round(
    (claims['claim_status_cleaned'] == "Approved").mean() * 100,
    2
)

pending_rate = round(
    (claims['claim_status_cleaned'] == "Pending").mean() * 100,
    2
)

denial_rate = round(
    (claims['claim_status_cleaned'] == "Denied").mean() * 100,
    2
)

partial_rate = round(
    (claims['claim_status_cleaned'] == "Partial").mean() * 100,
    2
)

print("Approval rate:", approval_rate)
print("Denial rate:", denial_rate)
print("Pending rate:", pending_rate)
print("Partial rate:", partial_rate)


Approval rate: 50.12
Denial rate: 20.02
Pending rate: 19.88
Partial rate: 9.98


In [40]:
# Total value for payes
claims['payer_cleaned'].value_counts()

payer_cleaned
Medicaid        28692
Cigna           28630
Aetna           28608
Medicare        28591
Self-Pay        28582
Unitedhealth    28532
Bluecross       28365
Name: count, dtype: Int64